# FinSight AI — Research Notebook 03: Machine Learning Model Benchmarking
## Comparative Evaluation: Linear Regression vs. Random Forest vs. XGBoost

In [ ]:
import sys
sys.path.append('../backend')
import yfinance as yf
import pandas as pd
from app.ml.feature_engineering import compute_features, create_target, time_series_split, FEATURE_COLUMNS
from app.ml.models import LinearRegressionModel, RandomForestModel, XGBoostModel
from app.ml.evaluation import evaluate_model

df = yf.download("NVDA", period="5y", interval="1d")[["Open", "High", "Low", "Close", "Volume"]].dropna()
feats = compute_features(df)
X, y = create_target(feats, horizon=5)
X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(X[FEATURE_COLUMNS], y)

In [ ]:
models = {
    "Linear Regression": LinearRegressionModel("NVDA", horizon=5),
    "Random Forest": RandomForestModel("NVDA", horizon=5),
    "XGBoost": XGBoostModel("NVDA", horizon=5),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train, X_val, y_val)
    preds = m.predict(X_test)
    results[name] = evaluate_model(y_test.values, preds, "test")

pd.DataFrame(results).T[["mae", "rmse", "mape", "r_squared", "directional_accuracy"]]